# Projeto Redes Neurais

In [ ]:
import pickle
from pathlib import Path

import mlx.core as mx
import mlx.nn as mnn
import mlx.optimizers as optim
import numpy as np
import pandas as pd
from sklearn.metrics import accuracy_score, classification_report
from sklearn.model_selection import StratifiedKFold
from tqdm.auto import tqdm
from xgboost import XGBClassifier

RANDOM_STATE = 42
N_SPLITS = 5
DATASETS_PICKLE_PATH = Path("data/datasets.pkl")

/Users/lucasreis/Documents/projeto-redes-neurais/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Treinamento

In [9]:
with open(DATASETS_PICKLE_PATH, "rb") as f:
    datasets = pickle.load(f)

print(f"datasets carregado de: {DATASETS_PICKLE_PATH}")
print("chaves disponíveis:", list(datasets.keys()))

datasets carregado de: data/datasets.pkl
chaves disponíveis: ['adult income', 'student', 'statlog', 'maintenance', 'stroke']


### GXboost

In [5]:
X_train = datasets["adult income"]["X_train"]
y_train = datasets["adult income"]["y_train"]

X_test = datasets["adult income"]["X_test"]
y_test = datasets["adult income"]["y_test"]

modelo_xgb = XGBClassifier(
    n_estimators=100,
    learning_rate=0.1,
    max_depth=6,
    random_state=42,
    eval_metric='logloss'
)

print("Iniciando o treinamento do XGBoost...")
modelo_xgb.fit(X_train, y_train)
print("Treinamento concluído com sucesso!")

y_pred = modelo_xgb.predict(X_test)

acuracia = accuracy_score(y_test, y_pred)

print("\n" + "="*60)
print("RELATÓRIO TÉCNICO DE AVALIAÇÃO DO MODELO (XGBOOST)")
print("="*60)
print(f"Acurácia Geral do Modelo: {acuracia * 100:.2f}%")
print("-"*60)
print(classification_report(y_test, y_pred))
print("="*60)

Iniciando o treinamento do XGBoost...
Treinamento concluído com sucesso!

RELATÓRIO TÉCNICO DE AVALIAÇÃO DO MODELO (XGBOOST)
Acurácia Geral do Modelo: 87.72%
------------------------------------------------------------
              precision    recall  f1-score   support

         0.0       0.90      0.95      0.92      7431
         1.0       0.80      0.65      0.72      2338

    accuracy                           0.88      9769
   macro avg       0.85      0.80      0.82      9769
weighted avg       0.87      0.88      0.87      9769



In [8]:
X_train = datasets["adult income"]["X_train_enriched"]
y_train = datasets["adult income"]["y_train_enriched"]

X_test = datasets["adult income"]["X_test"]
y_test = datasets["adult income"]["y_test"]

modelo_xgb = XGBClassifier(
    n_estimators=100,
    learning_rate=0.1,
    max_depth=6,
    random_state=42,
    eval_metric='logloss'
)

print("Iniciando o treinamento do XGBoost...")
modelo_xgb.fit(X_train, y_train)
print("Treinamento concluído com sucesso!")

y_pred = modelo_xgb.predict(X_test)

acuracia = accuracy_score(y_test, y_pred)

print("\n" + "="*60)
print("RELATÓRIO TÉCNICO DE AVALIAÇÃO DO MODELO (XGBOOST)")
print("="*60)
print(f"Acurácia Geral do Modelo: {acuracia * 100:.2f}%")
print("-"*60)
print(classification_report(y_test, y_pred))
print("="*60)

Iniciando o treinamento do XGBoost...
Treinamento concluído com sucesso!

RELATÓRIO TÉCNICO DE AVALIAÇÃO DO MODELO (XGBOOST)
Acurácia Geral do Modelo: 87.37%
------------------------------------------------------------
              precision    recall  f1-score   support

         0.0       0.89      0.95      0.92      7431
         1.0       0.79      0.64      0.71      2338

    accuracy                           0.87      9769
   macro avg       0.84      0.79      0.81      9769
weighted avg       0.87      0.87      0.87      9769



### NCTD

In [ ]:
def adaptive_max_pool2d(x, output_size):
    out_h, out_w = output_size
    in_h, in_w = x.shape[1], x.shape[2]

    stride_h = max(1, in_h // out_h)
    stride_w = max(1, in_w // out_w)
    kernel_h = in_h - (out_h - 1) * stride_h
    kernel_w = in_w - (out_w - 1) * stride_w

    pool = mnn.MaxPool2d(
        kernel_size=(kernel_h, kernel_w),
        stride=(stride_h, stride_w),
        padding=0,
    )
    return pool(x)


class SEBlock(mnn.Module):
    def __init__(self, channels, reduction=16):
        super().__init__()
        hidden = max(1, channels // reduction)
        self.fc1 = mnn.Linear(channels, hidden)
        self.fc2 = mnn.Linear(hidden, channels)

    def __call__(self, x):
        weights = mx.mean(x, axis=(1, 2))
        weights = mnn.relu(self.fc1(weights))
        weights = mx.sigmoid(self.fc2(weights))
        weights = mx.reshape(weights, (x.shape[0], 1, 1, x.shape[-1]))
        return x * weights


class ConvBlock(mnn.Module):
    def __init__(
        self,
        in_channels,
        out_channels,
        stride,
        target_size,
        use_se=False,
        use_asymmetric=False,
        use_dilation=False,
        use_residual=False,
    ):
        super().__init__()
        self.target_size = target_size
        self.use_se = use_se
        self.use_asymmetric = use_asymmetric
        self.use_residual = use_residual

        dilation = 2 if use_dilation else 1
        padding = dilation

        if use_asymmetric:
            self.conv_a = mnn.Conv2d(
                in_channels,
                out_channels,
                kernel_size=(1, 3),
                stride=(1, stride),
                padding=(0, padding),
                dilation=(1, dilation),
            )
            self.conv_b = mnn.Conv2d(
                out_channels,
                out_channels,
                kernel_size=(3, 1),
                stride=(stride, 1),
                padding=(padding, 0),
                dilation=(dilation, 1),
            )
        else:
            self.conv = mnn.Conv2d(
                in_channels,
                out_channels,
                kernel_size=3,
                stride=stride,
                padding=padding,
                dilation=dilation,
            )

        self.bn = mnn.BatchNorm(out_channels)
        self.se = SEBlock(out_channels) if use_se else None

        needs_projection = use_residual and (in_channels != out_channels or stride != 1)
        self.proj = (
            mnn.Conv2d(in_channels, out_channels, kernel_size=1, stride=stride)
            if needs_projection
            else None
        )

    def __call__(self, x):
        shortcut = x

        if self.use_asymmetric:
            x = self.conv_a(x)
            x = mnn.relu(x)
            x = self.conv_b(x)
        else:
            x = self.conv(x)

        x = self.bn(x)
        x = mnn.relu(x)

        if self.se is not None:
            x = self.se(x)

        if self.use_residual:
            if self.proj is not None:
                shortcut = self.proj(shortcut)
            x = x + shortcut

        x = adaptive_max_pool2d(x, self.target_size)
        return x


class NCTD_CNN_Base(mnn.Module):
    def __init__(
        self,
        use_se=False,
        use_asymmetric=False,
        use_dilation=False,
        use_residual=False,
    ):
        super().__init__()

        self.block0 = ConvBlock(
            1,
            64,
            stride=1,
            target_size=(15, 15),
            use_se=use_se,
            use_asymmetric=use_asymmetric,
            use_dilation=False,
            use_residual=use_residual,
        )
        self.block1 = ConvBlock(
            64,
            64,
            stride=2,
            target_size=(7, 7),
            use_se=use_se,
            use_asymmetric=use_asymmetric,
            use_dilation=use_dilation,
            use_residual=use_residual,
        )
        self.block2 = ConvBlock(
            64,
            64,
            stride=2,
            target_size=(3, 3),
            use_se=use_se,
            use_asymmetric=use_asymmetric,
            use_dilation=use_dilation,
            use_residual=use_residual,
        )
        self.block3 = ConvBlock(
            64,
            64,
            stride=2,
            target_size=(1, 1),
            use_se=use_se,
            use_asymmetric=use_asymmetric,
            use_dilation=False,
            use_residual=use_residual,
        )

        self.fc = mnn.Linear(64, 32)
        self.out = mnn.Linear(32, 2)

    def __call__(self, x):
        x = self.block0(x)
        x = self.block1(x)
        x = self.block2(x)
        x = self.block3(x)
        x = mx.mean(x, axis=(1, 2))
        x = mnn.relu(self.fc(x))
        x = self.out(x)
        return x


MODEL_NAME = "NCTD Base"
MODEL_CLASS = lambda: NCTD_CNN_Base()

In [ ]:
X_images_bruto = datasets["adult income"]["X_images"]

y_df = datasets["adult income"]["y"]
y_numeric = y_df.values if isinstance(y_df, pd.Series) else y_df
y_numeric = np.asarray(y_numeric, dtype=np.int32)

# MLX usa channels-last: (N, H, W, C)
X_images_mlx = np.expand_dims(X_images_bruto, axis=-1).astype(np.float32) / 255.0

BATCH_SIZE = 64
EPOCHS = 30

print(f"Dataset completo: {len(X_images_mlx)} amostras")
print(f"Folds: {N_SPLITS}")
print(f"Batch size: {BATCH_SIZE}")
print(f"Épocas por fold: {EPOCHS}")


def run_nctd_experiment(model_name, model_class, epochs=EPOCHS, n_splits=N_SPLITS):
    skf = StratifiedKFold(
        n_splits=n_splits,
        shuffle=True,
        random_state=RANDOM_STATE,
    )

    resultados_folds = []
    todas_predicoes = []
    todos_alvos = []

    print(f"\nModelo ativo: {model_name}")
    print(f"Validação cruzada estratificada com {n_splits} folds")

    for fold, (train_idx, val_idx) in enumerate(skf.split(X_images_mlx, y_numeric), start=1):
        X_train = X_images_mlx[train_idx]
        X_val = X_images_mlx[val_idx]
        y_train = y_numeric[train_idx]
        y_val = y_numeric[val_idx]

        num_batches_train = (len(X_train) + BATCH_SIZE - 1) // BATCH_SIZE
        num_batches_val = (len(X_val) + BATCH_SIZE - 1) // BATCH_SIZE
        total_steps = epochs * num_batches_train + num_batches_val

        model = model_class()
        mx.eval(model.parameters())

        optimizer = optim.Adam(learning_rate=0.0008)

        def loss_fn(model, x, y):
            logits = model(x)
            return mx.mean(mnn.losses.cross_entropy(logits, y))

        loss_and_grad_fn = mnn.value_and_grad(model, loss_fn)
        rng = np.random.default_rng(RANDOM_STATE + fold)

        print("\n" + "-" * 60)
        print(f"Fold {fold}/{n_splits}")
        print(f"Treino: {len(X_train)} amostras | Validação: {len(X_val)} amostras")
        print(f"Treinamento: {epochs} épocas × {num_batches_train} batches = {epochs * num_batches_train} iterações")
        print(f"Validação: {num_batches_val} iterações")

        with tqdm(total=total_steps, desc=f"{model_name} | fold {fold}", unit="iter") as pbar:
            for epoch in range(epochs):
                model.train()
                running_loss = 0.0

                perm = rng.permutation(len(X_train))

                for start in range(0, len(X_train), BATCH_SIZE):
                    end = min(start + BATCH_SIZE, len(X_train))
                    idx = perm[start:end]

                    batch_x = mx.array(X_train[idx])
                    batch_y = mx.array(y_train[idx])

                    loss, grads = loss_and_grad_fn(model, batch_x, batch_y)
                    optimizer.update(model, grads)
                    mx.eval(model.parameters(), optimizer.state)

                    loss_val = loss.item()
                    running_loss += loss_val * len(idx)

                    pbar.update(1)
                    pbar.set_postfix({
                        "epoch": f"{epoch + 1}/{epochs}",
                        "loss": f"{loss_val:.4f}",
                    })

                epoch_loss = running_loss / len(X_train)
                pbar.set_postfix({
                    "epoch": f"{epoch + 1}/{epochs} finalizada",
                    "loss": f"{epoch_loss:.4f}",
                })

            model.eval()
            predicoes_fold = []
            alvos_fold = []

            for start in range(0, len(X_val), BATCH_SIZE):
                end = min(start + BATCH_SIZE, len(X_val))
                batch_x = mx.array(X_val[start:end])

                logits = model(batch_x)
                preds = mx.argmax(logits, axis=1)
                mx.eval(preds)

                predicoes_fold.extend(np.array(preds).tolist())
                alvos_fold.extend(y_val[start:end].tolist())

                pbar.update(1)
                pbar.set_postfix({"fase": "validação"})

            pbar.set_postfix({"fase": "concluído"})

        acuracia_fold = accuracy_score(alvos_fold, predicoes_fold)
        resultados_folds.append({"fold": fold, "accuracy": acuracia_fold})
        todos_alvos.extend(alvos_fold)
        todas_predicoes.extend(predicoes_fold)

        print(f"Acurácia do fold {fold}: {acuracia_fold * 100:.2f}%")

    acuracias = np.array([resultado["accuracy"] for resultado in resultados_folds])
    acuracia_media = acuracias.mean()
    desvio_padrao = acuracias.std(ddof=1) if len(acuracias) > 1 else 0.0

    print("\n" + "=" * 60)
    print(f"RELATÓRIO TÉCNICO DE AVALIAÇÃO DA CNN ({model_name} - K-FOLD)")
    print("=" * 60)
    print(f"Acurácia média nas imagens 2N: {acuracia_media * 100:.2f}%")
    print(f"Desvio padrão: {desvio_padrao * 100:.2f} p.p.")
    print("Acurácias por fold: " + ", ".join(f"{acc * 100:.2f}%" for acc in acuracias))
    print("-" * 60)
    print(classification_report(todos_alvos, todas_predicoes, zero_division=0))
    print("=" * 60)

    return {
        "model_name": model_name,
        "folds": resultados_folds,
        "accuracy_mean": acuracia_media,
        "accuracy_std": desvio_padrao,
        "y_true": todos_alvos,
        "y_pred": todas_predicoes,
    }


In [ ]:
MODEL_NAME = "NCTD Attention (SE)"
MODEL_CLASS = lambda: NCTD_CNN_Base(use_se=True)
print(f"Modelo selecionado: {MODEL_NAME}")

Modelo selecionado: NCTD Attention (SE)


In [ ]:
resultado_attention = run_nctd_experiment(
    "NCTD Attention (SE)",
    lambda: NCTD_CNN_Base(use_se=True),
)


Modelo ativo: NCTD Attention (SE)
Treinamento: 30 épocas × 611 batches = 18330 iterações
Validação: 153 iterações
Total: 18483 iterações



NCTD Attention (SE): 100%|██████████| 18483/18483 [07:11<00:00, 42.81iter/s, fase=concluído]                     



RELATÓRIO TÉCNICO DE AVALIAÇÃO DA CNN (NCTD Attention (SE))
Acurácia Geral do Modelo nas Imagens 2N: 84.99%
------------------------------------------------------------
              precision    recall  f1-score   support

           0       0.89      0.92      0.90      7431
           1       0.71      0.62      0.66      2338

    accuracy                           0.85      9769
   macro avg       0.80      0.77      0.78      9769
weighted avg       0.84      0.85      0.85      9769



#### Variante 1 — Attention (SE)

In [ ]:
MODEL_NAME = "NCTD Asymmetric Kernels"
MODEL_CLASS = lambda: NCTD_CNN_Base(use_asymmetric=True)
print(f"Modelo selecionado: {MODEL_NAME}")

Modelo selecionado: NCTD Asymmetric Kernels


In [ ]:
resultado_asymmetric = run_nctd_experiment(
    "NCTD Asymmetric Kernels",
    lambda: NCTD_CNN_Base(use_asymmetric=True),
)


Modelo ativo: NCTD Asymmetric Kernels
Treinamento: 30 épocas × 611 batches = 18330 iterações
Validação: 153 iterações
Total: 18483 iterações



NCTD Asymmetric Kernels: 100%|██████████| 18483/18483 [07:18<00:00, 42.17iter/s, fase=concluído]                     



RELATÓRIO TÉCNICO DE AVALIAÇÃO DA CNN (NCTD Asymmetric Kernels)
Acurácia Geral do Modelo nas Imagens 2N: 85.09%
------------------------------------------------------------
              precision    recall  f1-score   support

           0       0.86      0.96      0.91      7431
           1       0.79      0.52      0.62      2338

    accuracy                           0.85      9769
   macro avg       0.82      0.74      0.77      9769
weighted avg       0.84      0.85      0.84      9769



#### Variante 2 — Kernels Assimétricos

In [ ]:
MODEL_NAME = "NCTD Dilated Convolutions"
MODEL_CLASS = lambda: NCTD_CNN_Base(use_dilation=True)
print(f"Modelo selecionado: {MODEL_NAME}")

Modelo selecionado: NCTD Dilated Convolutions


In [ ]:
resultado_dilated = run_nctd_experiment(
    "NCTD Dilated Convolutions",
    lambda: NCTD_CNN_Base(use_dilation=True),
)


Modelo ativo: NCTD Dilated Convolutions
Treinamento: 30 épocas × 611 batches = 18330 iterações
Validação: 153 iterações
Total: 18483 iterações



NCTD Dilated Convolutions: 100%|██████████| 18483/18483 [06:37<00:00, 46.46iter/s, fase=concluído]                     



RELATÓRIO TÉCNICO DE AVALIAÇÃO DA CNN (NCTD Dilated Convolutions)
Acurácia Geral do Modelo nas Imagens 2N: 84.40%
------------------------------------------------------------
              precision    recall  f1-score   support

           0       0.90      0.89      0.90      7431
           1       0.67      0.69      0.68      2338

    accuracy                           0.84      9769
   macro avg       0.79      0.79      0.79      9769
weighted avg       0.85      0.84      0.84      9769



#### Variante 3 — Convoluções Dilatadas

In [ ]:
MODEL_NAME = "NCTD Residual"
MODEL_CLASS = lambda: NCTD_CNN_Base(use_residual=True)
print(f"Modelo selecionado: {MODEL_NAME}")

Modelo selecionado: NCTD Residual


In [ ]:
resultado_residual = run_nctd_experiment(
    "NCTD Residual",
    lambda: NCTD_CNN_Base(use_residual=True),
)


Modelo ativo: NCTD Residual
Treinamento: 30 épocas × 611 batches = 18330 iterações
Validação: 153 iterações
Total: 18483 iterações



NCTD Residual: 100%|██████████| 18483/18483 [07:01<00:00, 43.81iter/s, fase=concluído]                     



RELATÓRIO TÉCNICO DE AVALIAÇÃO DA CNN (NCTD Residual)
Acurácia Geral do Modelo nas Imagens 2N: 84.84%
------------------------------------------------------------
              precision    recall  f1-score   support

           0       0.86      0.95      0.91      7431
           1       0.78      0.51      0.62      2338

    accuracy                           0.85      9769
   macro avg       0.82      0.73      0.76      9769
weighted avg       0.84      0.85      0.84      9769



#### Variante 4 — Conexões Residuais

In [ ]:
MODEL_NAME = "NCTD Base"
MODEL_CLASS = lambda: NCTD_CNN_Base()
print(f"Modelo selecionado: {MODEL_NAME}")

Modelo selecionado: NCTD Base


In [ ]:
resultado_base = run_nctd_experiment(
    "NCTD Base",
    lambda: NCTD_CNN_Base(),
)


Modelo ativo: NCTD Base
Treinamento: 30 épocas × 611 batches = 18330 iterações
Validação: 153 iterações
Total: 18483 iterações



NCTD Base: 100%|██████████| 18483/18483 [06:37<00:00, 46.53iter/s, fase=concluído]                     



RELATÓRIO TÉCNICO DE AVALIAÇÃO DA CNN (NCTD Base)
Acurácia Geral do Modelo nas Imagens 2N: 84.98%
------------------------------------------------------------
              precision    recall  f1-score   support

           0       0.87      0.94      0.91      7431
           1       0.75      0.55      0.64      2338

    accuracy                           0.85      9769
   macro avg       0.81      0.75      0.77      9769
weighted avg       0.84      0.85      0.84      9769



In [ ]:
NOISE_STD = 0.10


def apply_gaussian_noise_on_batch(batch_x, noise_std=NOISE_STD):
    noise = mx.random.normal(batch_x.shape, dtype=batch_x.dtype) * noise_std
    batch_x_noisy = batch_x + noise
    return mx.clip(batch_x_noisy, 0.0, 1.0)


modelo_com_ruido = NCTD_CNN_Base()
mx.eval(modelo_com_ruido.parameters())

optimizer_noise_aug = optim.Adam(learning_rate=0.0008)


def loss_fn_noise(model, x, y):
    logits = model(x)
    return mx.mean(mnn.losses.cross_entropy(logits, y))


loss_and_grad_fn_noise = mnn.value_and_grad(modelo_com_ruido, loss_fn_noise)
total_steps_noise = EPOCHS * num_batches_train + num_batches_val

print(f"\nModelo ativo: NCTD Base + Gaussian Noise Augmentation")
print(f"Ruído gaussiano: std = {NOISE_STD}")
print(f"Treinamento: {EPOCHS} épocas × {num_batches_train} batches = {EPOCHS * num_batches_train} iterações")
print(f"Validação: {num_batches_val} iterações (sem ruído)")
print(f"Total: {total_steps_noise} iterações\n")

with tqdm(total=total_steps_noise, desc="NCTD + Gaussian Noise", unit="iter") as pbar:
    for epoch in range(EPOCHS):
        modelo_com_ruido.train()
        running_loss = 0.0

        perm = np.random.permutation(len(X_train))

        for start in range(0, len(X_train), BATCH_SIZE):
            end = min(start + BATCH_SIZE, len(X_train))
            idx = perm[start:end]

            batch_x = mx.array(X_train[idx])
            batch_y = mx.array(y_train[idx])
            batch_x_noisy = apply_gaussian_noise_on_batch(batch_x)

            loss, grads = loss_and_grad_fn_noise(modelo_com_ruido, batch_x_noisy, batch_y)
            optimizer_noise_aug.update(modelo_com_ruido, grads)
            mx.eval(modelo_com_ruido.parameters(), optimizer_noise_aug.state)

            loss_val = loss.item()
            running_loss += loss_val * len(idx)

            pbar.update(1)
            pbar.set_postfix({
                "epoch": f"{epoch+1}/{EPOCHS}",
                "loss": f"{loss_val:.4f}",
                "noise_std": f"{NOISE_STD:.2f}",
            })

        epoch_loss = running_loss / len(X_train)
        pbar.set_postfix({
            "epoch": f"{epoch+1}/{EPOCHS} finalizada",
            "loss": f"{epoch_loss:.4f}",
            "noise_std": f"{NOISE_STD:.2f}",
        })

    modelo_com_ruido.eval()
    todas_predicoes_noise = []
    todos_alvos_noise = []

    for start in range(0, len(X_val), BATCH_SIZE):
        end = min(start + BATCH_SIZE, len(X_val))
        batch_x = mx.array(X_val[start:end])

        logits = modelo_com_ruido(batch_x)
        preds = mx.argmax(logits, axis=1)
        mx.eval(preds)

        todas_predicoes_noise.extend(np.array(preds).tolist())
        todos_alvos_noise.extend(y_val[start:end].tolist())

        pbar.update(1)
        pbar.set_postfix({"fase": "validação limpa", "noise_std": f"{NOISE_STD:.2f}"})

    pbar.set_postfix({"fase": "concluído", "noise_std": f"{NOISE_STD:.2f}"})

acuracia_noise_aug = accuracy_score(todos_alvos_noise, todas_predicoes_noise)

print("\n" + "=" * 60)
print("RELATÓRIO TÉCNICO DE AVALIAÇÃO DA CNN (NCTD Base + Gaussian Noise)")
print("=" * 60)
print(f"Acurácia Geral do Modelo nas Imagens 2N: {acuracia_noise_aug * 100:.2f}%")
print("-" * 60)
print(classification_report(todos_alvos_noise, todas_predicoes_noise))
print("=" * 60)

resultado_noise_aug = {
    "model_name": "NCTD Base + Gaussian Noise",
    "accuracy": acuracia_noise_aug,
    "y_true": todos_alvos_noise,
    "y_pred": todas_predicoes_noise,
    "noise_std": NOISE_STD,
}


Modelo ativo: NCTD Base + Gaussian Noise Augmentation
Ruído gaussiano: std = 0.1
Treinamento: 30 épocas × 611 batches = 18330 iterações
Validação: 153 iterações (sem ruído)
Total: 18483 iterações



NCTD + Gaussian Noise: 100%|██████████| 18483/18483 [06:50<00:00, 45.04iter/s, fase=concluído, noise_std=0.10]                     



RELATÓRIO TÉCNICO DE AVALIAÇÃO DA CNN (NCTD Base + Gaussian Noise)
Acurácia Geral do Modelo nas Imagens 2N: 83.54%
------------------------------------------------------------
              precision    recall  f1-score   support

           0       0.84      0.97      0.90      7431
           1       0.80      0.42      0.55      2338

    accuracy                           0.84      9769
   macro avg       0.82      0.69      0.72      9769
weighted avg       0.83      0.84      0.82      9769



#### Experimento Extra — Data Augmentation com Ruído Gaussiano (LM-IGTD inspirado)

#### Baseline — NCTD Base